In [1]:
%matplotlib qt
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits
import glob
from reprojection import *
from utils import *
from interpolation import *
import pandas as pd

In [2]:
files = sorted(glob.glob('/home/ulyanov/data/cross/*'))
files

['/home/ulyanov/data/cross/hmi.M_45s.20241120_001500_TAI.2.magnetogram.fits',
 '/home/ulyanov/data/cross/hmi.V_45s.20241120_001500_TAI.2.Dopplergram.fits',
 '/home/ulyanov/data/cross/solo_L2_phi-fdt-blos_20241120T001503_V202602220151_0451200501.fits.gz',
 '/home/ulyanov/data/cross/solo_L2_phi-fdt-vlos_20241120T001503_V202602220151_0451200501.fits.gz',
 '/home/ulyanov/data/cross/solo_L2_phi-fdt-vlos_20250501T002002_V202602220258_0545010501.fits.gz']

In [3]:
file = files[1]

with fits.open(file) as hdul:
    header = hdul[1].header
    data = hdul[1].data

data = rebin(data, 8, update_header=header)

In [4]:
view = View.from_header(header)

In [5]:
V = view.velocity(mu_thr=0.0) / 1000
V -= np.nanmean(V)

plt.figure(figsize=(10,10))
plt.imshow(V, 'seismic', vmin=-2, vmax=2)

plt.tight_layout()

In [6]:
V_ = data.copy() / 1000
V_[np.isnan(V)] = np.nan
V_ -= np.nanmean(V_)

plt.figure(figsize=(10,10))
plt.imshow(V_, 'seismic', vmin=-2, vmax=2)

plt.tight_layout()

In [7]:
t = ~np.isnan(V)
x, y = V[t], V_[t]
t = np.where(np.abs(y - x) < 0.5)
x, y = x[t], y[t]

k = np.nanmean(x * y) / np.nanmean(x ** 2)
print(k)

plt.figure(figsize=(10,10))
plt.plot([-2,2], [-2,2], color='black', lw=0.5)
plt.scatter(V, V_, s=0.01)
plt.plot([-2,2], [-2 * k, 2 * k], '--', color='black', lw=1)


plt.xlim(-2,2)
plt.ylim(-2,2)

plt.tight_layout()

0.9975001414781436


In [8]:
plt.figure(figsize=(10,10))
plt.imshow(V_ - V * k, 'seismic', vmin=-1, vmax=1)

plt.tight_layout()

In [20]:
q = 1e-9 * 180 / np.pi * 24 * 3600

T1 = 2300 * q
T3 = -54.5 * q
T5 = -6.5 * q

A = 1 / 16 * (8 * np.sqrt(6) * T1 - 12 * np.sqrt(14) * T3 + 15 * np.sqrt(22) * T5)
B = 15 / 8 * (2 * np.sqrt(14) * T3 - 7 * np.sqrt(22) * T5)
C = 315 / 8 * np.sqrt(11 / 2) * T5

A, B, C

(np.float64(14.560337763326853),
 np.float64(-1.8046527057290216),
 np.float64(-2.971335167231601))